# Gaia BP/RP Spectra Downloader with Catalog Enrichment (v1)

This notebook:
1. Reads `source_id` from any FITS file
2. **NEW**: Skips already-downloaded spectra (caching)
3. Checks which sources have BP/RP sampled spectra in Gaia DR3
4. Downloads them in batches to `./BPRP_spectra/<source_id>.fits`
5. **NEW**: Enriches catalog with Andrae+2023 parameters (Teff_A23, logg_A23, MH_A23)
6. **NEW**: Enriches catalog with Wang+2025 3D dust map (EBV_W25, A_V_W25)
7. **NEW**: Saves enriched catalog

## Setup and Imports

In [1]:
import numpy as np
from astropy.io import fits
from astropy.table import Table, Column
from astroquery.gaia import Gaia
from pathlib import Path
import os
import h5py

# Install dustmaps3d if needed
try:
    from dustmaps3d import dustmaps3d
    print("dustmaps3d already installed")
except ImportError:
    print("Installing dustmaps3d...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'dustmaps3d'])
    from dustmaps3d import dustmaps3d
    print("dustmaps3d installed successfully")

# Directory for downloaded spectra
spectra_dir = Path('./BPRP_spectra')
spectra_dir.mkdir(exist_ok=True)

print(f"Spectra will be saved to: {spectra_dir.absolute()}")

Workaround solutions for the Gaia Archive issues following the infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive
dustmaps3d already installed
Spectra will be saved to: /Users/rix/Science/Projects/GAIA/GaiaDR3/BP-RP/All-Sky/HotStars_XP/BPRP_spectra


## Configuration

In [5]:
# ========================== CONFIGURATION ==========================

# Input catalog
CATALOG_FITS = 'SB1Cands.3.fits'  # Change this to your input file

# Andrae+2023 XGBoost parameters file
A23_HDF5_FILE = '/Users/rix/Science/Projects/GAIA/GaiaDR3/BP-RP/All-Sky/table-1.hdf5'

# Output enriched catalog (will be saved after enrichment)
OUTPUT_ENRICHED_CATALOG = CATALOG_FITS.replace('.fits', '_enriched.fits')

# Caching: skip already-downloaded spectra?
SKIP_EXISTING_SPECTRA = True

# R_V for converting E(B-V) to A_V
R_V_FOR_AV_W25 = 3.1

print(f"Input catalog: {CATALOG_FITS}")
print(f"Output enriched catalog: {OUTPUT_ENRICHED_CATALOG}")
print(f"Skip existing spectra: {SKIP_EXISTING_SPECTRA}")
print(f"R_V for A_V_W25: {R_V_FOR_AV_W25}")

Input catalog: SB1Cands.3.fits
Output enriched catalog: SB1Cands.3_enriched.fits
Skip existing spectra: True
R_V for A_V_W25: 3.1


## Optional: Login to Gaia Archive

In [6]:
# Login to Gaia archive (optional - public data works without login)
# Uncomment and fill in if you have credentials
# username = 'your_username'
# password = 'your_password'
# try:
#     Gaia.login(user=username, password=password)
#     print("Successfully logged in to Gaia archive")
# except Exception as e:
#     print(f"Login failed: {e}")

print("Using anonymous access to Gaia archive (works for public DR3 data)")

Using anonymous access to Gaia archive (works for public DR3 data)


## Load source_id from FITS file

In [7]:
# Load catalog
with fits.open(CATALOG_FITS) as hdul:
    catalog_data = Table(hdul[1].data)
    source_ids = np.array(catalog_data['source_id'], dtype=np.int64)

# Get unique source_ids while preserving order
_, unique_idx = np.unique(source_ids, return_index=True)
unique_idx = np.sort(unique_idx)
source_ids_unique = source_ids[unique_idx]

print(f"Loaded {len(source_ids)} sources from {CATALOG_FITS}")
print(f"Unique source_ids: {len(source_ids_unique)}")
print(f"Columns in catalog: {catalog_data.colnames}")

Loaded 1258 sources from SB1Cands.3.fits
Unique source_ids: 1258
Columns in catalog: ['source_id', 'ra', 'dec', 'parallax', 'parallax_error', 'Gmag']


## Check for already-downloaded spectra (caching)

In [8]:
if SKIP_EXISTING_SPECTRA:
    # Find which spectra already exist
    existing_files = set(int(f.stem) for f in spectra_dir.glob('*.fits') if f.stem.isdigit())
    
    # Filter to only sources we don't have yet
    sources_to_download = np.array([sid for sid in source_ids_unique if sid not in existing_files])
    
    print(f"Already have {len(existing_files)} spectra downloaded")
    print(f"Need to download: {len(sources_to_download)} spectra")
    print(f"Skipping: {len(source_ids_unique) - len(sources_to_download)} existing spectra")
else:
    sources_to_download = source_ids_unique
    print(f"Will attempt to download all {len(sources_to_download)} spectra (caching disabled)")

Already have 88022 spectra downloaded
Need to download: 153 spectra
Skipping: 1105 existing spectra


## Check which sources have BP/RP sampled spectra

In [9]:
def check_bprp_availability(source_ids, batch_size=2000):
    """Check which sources have BP/RP sampled spectra in Gaia DR3."""
    sources_with_spectra = []
    print("Checking which sources have BP/RP sampled spectra...")
    
    for i in range(0, len(source_ids), batch_size):
        batch = source_ids[i:i+batch_size]
        id_list = ','.join(map(str, batch))
        
        query = f"""
        SELECT source_id
        FROM gaiadr3.gaia_source
        WHERE source_id IN ({id_list})
          AND has_xp_sampled = 'true'
        """
        
        try:
            job = Gaia.launch_job(query)
            result = job.get_results()
            sources_with_spectra.extend(result['source_id'].data)
            print(f"  Batch {i//batch_size + 1}/{(len(source_ids)-1)//batch_size + 1}: "
                  f"{len(result)} sources have sampled spectra")
        except Exception as e:
            print(f"  Batch failed (will skip): {e}")
    
    return np.array(sources_with_spectra, dtype=np.int64)

# Only check for spectra we need to download
if len(sources_to_download) > 0:
    sources_with_spectra = check_bprp_availability(sources_to_download)
    print(f"\nDone: {len(sources_with_spectra)} / {len(sources_to_download)} "
          f"sources have BP/RP sampled spectra ({100*len(sources_with_spectra)/len(sources_to_download):.1f}%)")
else:
    sources_with_spectra = np.array([], dtype=np.int64)
    print("\nNo new spectra to download!")

Checking which sources have BP/RP sampled spectra...
  Batch 1/1: 0 sources have sampled spectra

Done: 0 / 153 sources have BP/RP sampled spectra (0.0%)


## Download BP/RP spectra in batches

In [10]:
if len(sources_with_spectra) > 0:
    batch_size = 50
    n_batches = (len(sources_with_spectra) + batch_size - 1) // batch_size

    print(f"\nDownloading {len(sources_with_spectra)} spectra in {n_batches} batches of ~{batch_size}...\n")

    for batch_idx in range(0, len(sources_with_spectra), batch_size):
        batch_ids = sources_with_spectra[batch_idx:batch_idx + batch_size]
        print(f"Batch {(batch_idx//batch_size)+1}/{n_batches} - downloading {len(batch_ids)} sources...")
        
        try:
            datalink = Gaia.load_data(ids=batch_ids, retrieval_type='XP_SAMPLED', 
                                       data_structure='INDIVIDUAL', verbose=False)
            
            for key, prod_list in datalink.items():
                source_id = None
                for sid in batch_ids:
                    if str(sid) in key:
                        source_id = sid
                        break
                if source_id is None:
                    continue
                    
                prod = prod_list[0] if isinstance(prod_list, list) else prod_list
                table = prod.to_table() if hasattr(prod, 'to_table') else prod
                
                out_file = spectra_dir / f"{source_id}.fits"
                table.write(out_file, overwrite=True)
                
        except Exception as e:
            print(f"  [Warning] Batch failed: {e}. Falling back to individual downloads...")
            for sid in batch_ids:
                try:
                    dl = Gaia.load_data(ids=[sid], retrieval_type='XP_SAMPLED', 
                                        data_structure='INDIVIDUAL', verbose=False)
                    for k, pl in dl.items():
                        if str(sid) not in k:
                            continue
                        prod = pl[0] if isinstance(pl, list) else pl
                        tab = prod.to_table() if hasattr(prod, 'to_table') else prod
                        out_file = spectra_dir / f"{sid}.fits"
                        tab.write(out_file, overwrite=True)
                        break
                except Exception as e2:
                    print(f"    [Error] Failed {sid}: {e2}")

    print(f"\nDownload complete! Spectra saved in: {spectra_dir}")
else:
    print("No spectra to download.")

No spectra to download.


---
# Catalog Enrichment

Now we add external information to the catalog:
1. Andrae+2023 XGBoost stellar parameters (Teff, logg, [M/H])
2. Wang+2025 3D dust map E(B-V) and derived A_V

## Load Andrae+2023 Parameters

In [11]:
print(f"Loading Andrae+2023 parameters from {A23_HDF5_FILE}...")
print("(This may take a minute for 124M rows)")

# Load the HDF5 file and build a lookup dictionary
# The file has: source_id, teff_xgboost, logg_xgboost, mh_xgboost (all as strings)

with h5py.File(A23_HDF5_FILE, 'r') as f:
    a23_source_ids = np.array([int(s.decode()) for s in f['source_id'][:]])
    a23_teff = np.array([float(s.decode()) if s.decode().strip() else np.nan 
                         for s in f['teff_xgboost'][:]])
    a23_logg = np.array([float(s.decode()) if s.decode().strip() else np.nan 
                         for s in f['logg_xgboost'][:]])
    a23_mh = np.array([float(s.decode()) if s.decode().strip() else np.nan 
                       for s in f['mh_xgboost'][:]])

print(f"Loaded {len(a23_source_ids)} A23 entries")

# Build lookup dictionary: source_id -> (Teff, logg, MH)
print("Building lookup dictionary...")
a23_lookup = {sid: (teff, logg, mh) 
              for sid, teff, logg, mh in zip(a23_source_ids, a23_teff, a23_logg, a23_mh)}
print(f"Dictionary ready with {len(a23_lookup)} entries")

Loading Andrae+2023 parameters from /Users/rix/Science/Projects/GAIA/GaiaDR3/BP-RP/All-Sky/table-1.hdf5...
(This may take a minute for 124M rows)
Loaded 124060658 A23 entries
Building lookup dictionary...
Dictionary ready with 124060658 entries


In [12]:
# Match catalog sources to A23
print("\nMatching catalog sources to Andrae+2023...")

Teff_A23 = np.full(len(catalog_data), np.nan)
logg_A23 = np.full(len(catalog_data), np.nan)
MH_A23 = np.full(len(catalog_data), np.nan)

n_matched = 0
for i, sid in enumerate(source_ids):
    if sid in a23_lookup:
        Teff_A23[i], logg_A23[i], MH_A23[i] = a23_lookup[sid]
        n_matched += 1

print(f"Matched {n_matched} / {len(source_ids)} sources ({100*n_matched/len(source_ids):.1f}%)")
print(f"Teff_A23 range: {np.nanmin(Teff_A23):.0f} - {np.nanmax(Teff_A23):.0f} K")
print(f"logg_A23 range: {np.nanmin(logg_A23):.2f} - {np.nanmax(logg_A23):.2f}")
print(f"MH_A23 range: {np.nanmin(MH_A23):.2f} - {np.nanmax(MH_A23):.2f}")


Matching catalog sources to Andrae+2023...
Matched 946 / 1258 sources (75.2%)
Teff_A23 range: 3215 - 6969 K
logg_A23 range: 0.02 - 4.85
MH_A23 range: -2.94 - 0.23


## Query Wang+2025 3D Dust Map

In [13]:
print("Querying Wang+2025 3D dust map...")
print("(First run will download ~400MB data file)")

# Get coordinates and distances from catalog
# Need Galactic l, b and distance in kpc

# Check what coordinate columns we have
if 'GLON' in catalog_data.colnames and 'GLAT' in catalog_data.colnames:
    gal_l = np.array(catalog_data['GLON'])
    gal_b = np.array(catalog_data['GLAT'])
    print(f"Using GLON, GLAT from catalog")
elif 'l' in catalog_data.colnames and 'b' in catalog_data.colnames:
    gal_l = np.array(catalog_data['l'])
    gal_b = np.array(catalog_data['b'])
    print(f"Using l, b from catalog")
else:
    # Convert from RA, Dec to Galactic
    from astropy.coordinates import SkyCoord
    import astropy.units as u
    
    ra_col = 'RA_ICRS' if 'RA_ICRS' in catalog_data.colnames else 'ra'
    dec_col = 'DE_ICRS' if 'DE_ICRS' in catalog_data.colnames else 'dec'
    
    coords = SkyCoord(ra=catalog_data[ra_col]*u.deg, dec=catalog_data[dec_col]*u.deg, frame='icrs')
    galactic = coords.galactic
    gal_l = galactic.l.deg
    gal_b = galactic.b.deg
    print(f"Converted {ra_col}, {dec_col} to Galactic coordinates")

# Get distance in kpc
# Prefer parallax-based distance, but could use photometric distance if available
if 'Plx' in catalog_data.colnames:
    parallax_mas = np.array(catalog_data['Plx'])
elif 'parallax' in catalog_data.colnames:
    parallax_mas = np.array(catalog_data['parallax'])
else:
    raise ValueError("No parallax column found in catalog!")

# Convert parallax to distance in kpc (parallax in mas -> distance in pc -> kpc)
# Handle negative or zero parallax
with np.errstate(divide='ignore', invalid='ignore'):
    dist_kpc = np.where(parallax_mas > 0, 1.0 / parallax_mas, np.nan)  # 1/mas = kpc

print(f"Distance range: {np.nanmin(dist_kpc):.2f} - {np.nanmax(dist_kpc):.2f} kpc")
print(f"Valid distances: {np.sum(np.isfinite(dist_kpc))} / {len(dist_kpc)}")

Querying Wang+2025 3D dust map...
(First run will download ~400MB data file)
Converted ra, dec to Galactic coordinates
Distance range: 0.02 - 9.89 kpc
Valid distances: 1258 / 1258


In [14]:
# Query the dust map
# dustmaps3d returns: EBV, dust_density, sigma, max_d

# Only query for sources with valid positions and distances
valid_mask = np.isfinite(gal_l) & np.isfinite(gal_b) & np.isfinite(dist_kpc) & (dist_kpc > 0)
print(f"Querying dust map for {np.sum(valid_mask)} sources with valid coordinates...")

# Initialize output arrays
EBV_W25 = np.full(len(catalog_data), np.nan)
EBV_W25_err = np.full(len(catalog_data), np.nan)
dist_max_W25 = np.full(len(catalog_data), np.nan)

if np.sum(valid_mask) > 0:
    # Query in batches to show progress
    l_valid = gal_l[valid_mask]
    b_valid = gal_b[valid_mask]
    d_valid = dist_kpc[valid_mask]
    
    try:
        # dustmaps3d accepts arrays directly
        EBV_result, dust_result, sigma_result, maxd_result = dustmaps3d(l_valid, b_valid, d_valid)
        
        # Store results back into full arrays
        EBV_W25[valid_mask] = EBV_result.values if hasattr(EBV_result, 'values') else EBV_result
        EBV_W25_err[valid_mask] = sigma_result.values if hasattr(sigma_result, 'values') else sigma_result
        dist_max_W25[valid_mask] = maxd_result.values if hasattr(maxd_result, 'values') else maxd_result
        
        print(f"Dust map query complete!")
        print(f"E(B-V) range: {np.nanmin(EBV_W25):.3f} - {np.nanmax(EBV_W25):.3f} mag")
        print(f"E(B-V) uncertainty range: {np.nanmin(EBV_W25_err):.3f} - {np.nanmax(EBV_W25_err):.3f} mag")
        
    except Exception as e:
        print(f"Error querying dust map: {e}")
        print("Continuing with NaN values for dust map columns")
else:
    print("No valid coordinates for dust map query!")

Querying dust map for 1203 sources with valid coordinates...
Dust map query complete!
E(B-V) range: 0.000 - 1.430 mag
E(B-V) uncertainty range: 0.001 - 0.534 mag


In [15]:
# Convert E(B-V) to A_V using R_V = 3.1
A_V_W25 = R_V_FOR_AV_W25 * EBV_W25
A_V_W25_err = R_V_FOR_AV_W25 * EBV_W25_err

print(f"A_V_W25 = {R_V_FOR_AV_W25} x E(B-V)")
print(f"A_V_W25 range: {np.nanmin(A_V_W25):.3f} - {np.nanmax(A_V_W25):.3f} mag")
print(f"Median A_V_W25: {np.nanmedian(A_V_W25):.3f} mag")

A_V_W25 = 3.1 x E(B-V)
A_V_W25 range: 0.000 - 4.433 mag
Median A_V_W25: 1.184 mag


## Save Enriched Catalog

In [16]:
# Check if columns already exist (don't overwrite)
new_columns = []

if 'Teff_A23' not in catalog_data.colnames:
    catalog_data.add_column(Column(Teff_A23, name='Teff_A23'))
    new_columns.append('Teff_A23')
else:
    print("Teff_A23 already exists, skipping")

if 'logg_A23' not in catalog_data.colnames:
    catalog_data.add_column(Column(logg_A23, name='logg_A23'))
    new_columns.append('logg_A23')
else:
    print("logg_A23 already exists, skipping")

if 'MH_A23' not in catalog_data.colnames:
    catalog_data.add_column(Column(MH_A23, name='MH_A23'))
    new_columns.append('MH_A23')
else:
    print("MH_A23 already exists, skipping")

if 'EBV_W25' not in catalog_data.colnames:
    catalog_data.add_column(Column(EBV_W25, name='EBV_W25'))
    new_columns.append('EBV_W25')
else:
    print("EBV_W25 already exists, skipping")

if 'EBV_W25_err' not in catalog_data.colnames:
    catalog_data.add_column(Column(EBV_W25_err, name='EBV_W25_err'))
    new_columns.append('EBV_W25_err')
else:
    print("EBV_W25_err already exists, skipping")

if 'A_V_W25' not in catalog_data.colnames:
    catalog_data.add_column(Column(A_V_W25, name='A_V_W25'))
    new_columns.append('A_V_W25')
else:
    print("A_V_W25 already exists, skipping")

if 'A_V_W25_err' not in catalog_data.colnames:
    catalog_data.add_column(Column(A_V_W25_err, name='A_V_W25_err'))
    new_columns.append('A_V_W25_err')
else:
    print("A_V_W25_err already exists, skipping")

if 'dist_max_W25' not in catalog_data.colnames:
    catalog_data.add_column(Column(dist_max_W25, name='dist_max_W25'))
    new_columns.append('dist_max_W25')
else:
    print("dist_max_W25 already exists, skipping")

print(f"\nAdded {len(new_columns)} new columns: {new_columns}")


Added 8 new columns: ['Teff_A23', 'logg_A23', 'MH_A23', 'EBV_W25', 'EBV_W25_err', 'A_V_W25', 'A_V_W25_err', 'dist_max_W25']


In [17]:
# Save enriched catalog
catalog_data.write(OUTPUT_ENRICHED_CATALOG, overwrite=True)
print(f"Saved enriched catalog to: {OUTPUT_ENRICHED_CATALOG}")
print(f"Total columns: {len(catalog_data.colnames)}")
print(f"Columns: {catalog_data.colnames}")

Saved enriched catalog to: SB1Cands.3_enriched.fits
Total columns: 14
Columns: ['source_id', 'ra', 'dec', 'parallax', 'parallax_error', 'Gmag', 'Teff_A23', 'logg_A23', 'MH_A23', 'EBV_W25', 'EBV_W25_err', 'A_V_W25', 'A_V_W25_err', 'dist_max_W25']


## Summary Statistics

In [18]:
print("="*60)
print("ENRICHMENT SUMMARY")
print("="*60)
print(f"\nInput catalog: {CATALOG_FITS}")
print(f"Output catalog: {OUTPUT_ENRICHED_CATALOG}")
print(f"Total sources: {len(catalog_data)}")

print(f"\nAndrae+2023 (XGBoost) parameters:")
print(f"  Matched: {np.sum(np.isfinite(Teff_A23))} / {len(Teff_A23)} ({100*np.sum(np.isfinite(Teff_A23))/len(Teff_A23):.1f}%)")
print(f"  Teff_A23: {np.nanmin(Teff_A23):.0f} - {np.nanmax(Teff_A23):.0f} K (median {np.nanmedian(Teff_A23):.0f} K)")

print(f"\nWang+2025 3D dust map:")
print(f"  Valid E(B-V): {np.sum(np.isfinite(EBV_W25))} / {len(EBV_W25)} ({100*np.sum(np.isfinite(EBV_W25))/len(EBV_W25):.1f}%)")
print(f"  A_V_W25: {np.nanmin(A_V_W25):.3f} - {np.nanmax(A_V_W25):.3f} mag (median {np.nanmedian(A_V_W25):.3f} mag)")

print(f"\nSpectra downloaded: {len(list(spectra_dir.glob('*.fits')))} files in {spectra_dir}")
print("="*60)

ENRICHMENT SUMMARY

Input catalog: SB1Cands.3.fits
Output catalog: SB1Cands.3_enriched.fits
Total sources: 1258

Andrae+2023 (XGBoost) parameters:
  Matched: 946 / 1258 (75.2%)
  Teff_A23: 3215 - 6969 K (median 5606 K)

Wang+2025 3D dust map:
  Valid E(B-V): 1203 / 1258 (95.6%)
  A_V_W25: 0.000 - 4.433 mag (median 1.184 mag)

Spectra downloaded: 88022 files in BPRP_spectra


In [19]:
# Optional: Logout from Gaia
Gaia.logout()

INFO: Gaia TAP server logout OK [astroquery.gaia.core]
INFO: Gaia data server logout OK [astroquery.gaia.core]
